# M11 -- look at the samples; anchor Tier 3 to the paper (Kaggle T4)

M9 measured every Tier-3 solver by L2-to-reference alone and never rendered a
single image. On that metric, DPM-Solver-2 *loses* to DPM-Solver-1 at 10 NFE --
the opposite of the paper's own FID ranking (Table 6: DPM-2 7.90 < DPM-1 16.69
< DPM-3 24.37). A review (`docs/MILESTONES_M11-M13.md`, Finding 1) found no
code bug, but two real gaps: DPM-Solver-fast (the sampler the paper actually
uses at <= 20 NFE) was never run, and there is nothing to look at.

This notebook produces the images, reproduces the paper's Figure 4 on *our*
checkpoint, and runs a small FID anchor. It is a **decision gate**, not just
another figure:

- **FID ranking at 10 NFE matches Table 6** (DPM-fast <~ DPM-2 < DDIM ~ DPM-1 <
  DPM-3) -> Tier 3 is validated, and the L2-vs-FID disagreement becomes a
  headline result -- the project's own thesis ("judge solvers as ODE solvers,
  not by FID") made visible on one plot.
- **Doesn't match** -> stop before M12/M13's report rewrite. First suspects:
  the discrete-schedule conversion, and the continuous-to-discrete
  time-input mapping (see `src/tier3.py`'s module docstring).

---

### How to run this on Kaggle

1. New Notebook -> **Settings -> Accelerator: GPU T4 x1**
2. **Settings -> Internet: ON** (git clone, `from_pretrained`, and the FID
   reference-statistics download all need it)
3. Run All. Budget ~30-40 minutes at 5k FID samples (the notebook's default),
   roughly double at 10k -- see the `N_FID_SAMPLES` knob in section 3.
4. Download `results/tier3_fid.csv` and `figures/11_*.png` from
   `/kaggle/working/NUMERICAL-PROJECT/`, drop them into the repo, commit.

Needs `results/tier3_ref_s201_e1e-3.pt` from M9 (the 201-NFE reference) --
regenerated automatically if the cached file isn't present, at the cost of one
more ~2 minute solve. `clean-fid`'s own reference statistics are downloaded
and cached by the package itself on first use.

In [ ]:
import os, sys, pathlib, subprocess, time, shutil

IN_KAGGLE = pathlib.Path("/kaggle").exists()
REPO = "https://github.com/MehemudAzad/NUMERICAL-PROJECT.git"

if IN_KAGGLE:
    ROOT = pathlib.Path("/kaggle/working/NUMERICAL-PROJECT")
    if ROOT.exists():
        subprocess.run(["git", "fetch", "--depth", "1", "origin", "main"],
                       cwd=ROOT, check=True)
        subprocess.run(["git", "reset", "--hard", "origin/main"], cwd=ROOT, check=True)
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPO],
                       cwd="/kaggle/working", check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "diffusers", "accelerate", "clean-fid"], check=True)
else:
    p = pathlib.Path.cwd()
    while not (p / "pyproject.toml").exists() and p != p.parent:
        p = p.parent
    ROOT = p

sys.path.insert(0, str(ROOT))

import importlib

for _mod in [m for m in list(sys.modules)
             if m == "src" or m.startswith(("src.", "third_party"))]:
    del sys.modules[_mod]
importlib.invalidate_caches()

if not (ROOT / "src" / "imaging.py").exists():
    raise RuntimeError(
        f"src/imaging.py is not in {ROOT}. The M11 commit has not been pushed "
        "to origin/main yet -- this notebook clones the repo, so it can only "
        "see what is on GitHub. Fix: push locally, then re-run this cell."
    )

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

# Do NOT import src.arm_c here -- see src/tier3.py's module docstring. It sets
# torch.set_default_dtype(float64) as a module-level side effect, which would
# build the UNet in float64 and break the float32 solver grids.
from src.grids import grid_t
from src.metrics import l2
from src.imaging import to_uint8
from src.runlog import append_row, load
from src.solvers import integrate
from src.testbeds import pf_rhs_lambda, pf_rhs_t
from src.tier3 import (DiscreteSchedule, make_eps_fn, make_model_fn,
                       make_noise_schedule, sample_dpm_solver_t3)

RESULTS = ROOT / "results"
FIGURES = ROOT / "figures"
RESULTS.mkdir(exist_ok=True)
FIGURES.mkdir(exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"root   : {ROOT}")
print(f"device : {DEVICE}", f"({torch.cuda.get_device_name(0)})" if DEVICE == "cuda" else "")
assert torch.get_default_dtype() == torch.float32, "restart the kernel: something set float64"
if DEVICE == "cpu":
    print("\n!! No GPU. This notebook will run, but far too slowly for the FID section --")
    print("!! set Accelerator to T4 (Settings, top right).")


## 1. The checkpoint and the reference, reusing M9's cache

Same checkpoint, same schedule construction, same cached 201-NFE reference as
M9 (`notebooks/09_tier3_cifar10.ipynb`) -- everything here is compared against
the *same* fixed point M9 used, so a difference in this notebook's findings is
about the *metric*, not about a different setup.

In [ ]:
from diffusers import DDPMScheduler, UNet2DModel

MODEL_ID = "google/ddpm-cifar10-32"
SEED = 0
N_GRID_SAMPLES = 8     # Figure-4 grid: one row of images per sampler
N_L2_SAMPLES = 64      # part B: per-image L2 distribution, matches M9's batch

t0 = time.time()
unet = UNet2DModel.from_pretrained(MODEL_ID).to(DEVICE).eval()
ddpm = DDPMScheduler.from_pretrained(MODEL_ID)
print(f"checkpoint loaded in {time.time()-t0:.1f}s")

betas = ddpm.betas.double()
ns32 = make_noise_schedule(betas, dtype=torch.float32, device=DEVICE)
ns64 = make_noise_schedule(betas, dtype=torch.float64)
ds = DiscreteSchedule(ns64)

model_fn = make_model_fn(unet, ns32)
eps_fn = make_eps_fn(model_fn)

T, T_END = 1.0, ds.t_min

torch.manual_seed(SEED)
x_T_grid = torch.randn(N_GRID_SAMPLES, 3, 32, 32, device=DEVICE)
torch.manual_seed(SEED + 1)
x_T_l2 = torch.randn(N_L2_SAMPLES, 3, 32, 32, device=DEVICE)

ref_path = RESULTS / "tier3_ref_s201_e1e-3.pt"
if ref_path.exists():
    ref_l2_full = torch.load(ref_path, map_location=DEVICE)
    print("reference loaded from M9's cache:", ref_path.relative_to(ROOT))
    if ref_l2_full.shape[0] != N_L2_SAMPLES:
        print(f"!! cached reference has {ref_l2_full.shape[0]} samples, need {N_L2_SAMPLES} -- recomputing")
        ref_l2_full = None
else:
    ref_l2_full = None

if ref_l2_full is None:
    t0 = time.time()
    ref_l2_full, _ = sample_dpm_solver_t3(model_fn, ns32, x_T_l2, T, T_END, order=3, steps=201)
    torch.save(ref_l2_full.cpu(), ref_path)
    print(f"reference recomputed in {time.time()-t0:.1f}s")

# The grid's own reference, same x_T as the grid so Figure 4's rightmost column is valid.
ref_grid, _ = sample_dpm_solver_t3(model_fn, ns32, x_T_grid, T, T_END, order=3, steps=201)


## 2. Samplers

Six configurations, all from the checkpoint's own schedule, `algorithm_type="dpmsolver"`:

| Label | Call |
|---|---|
| DDIM (quadratic) | `order=1, skip_type="time_quadratic"` -- the paper's Figure 4 baseline |
| DPM-1 / DPM-2 / DPM-3 | `skip_type="logSNR"`, what M9 ran |
| DPM-Solver-fast | `method="singlestep"`, `order=3` -- mixes orders to spend exactly `steps` NFE |
| RK2 in t (arm A) | `integrate(pf_rhs_t, ..., "midpoint")` on `ds` |

RK4-in-lambda (arm B) is dropped from the image grid to keep it readable (12
columns is already a lot); it stays in the FID table below, since the
L2-vs-FID contrast is the point of including arms A/B at all.

In [ ]:
def run_sampler(label, x_T, nfe):
    if label == "ddim_quad":
        xf, real_nfe = sample_dpm_solver_t3(model_fn, ns32, x_T, T, T_END, order=1, steps=nfe,
                                            skip_type="time_quadratic")
    elif label in ("dpm1", "dpm2", "dpm3"):
        order = {"dpm1": 1, "dpm2": 2, "dpm3": 3}[label]
        steps = max(order, (nfe // order) * order)
        xf, real_nfe = sample_dpm_solver_t3(model_fn, ns32, x_T, T, T_END, order=order, steps=steps)
    elif label == "dpm_fast":
        xf, real_nfe = sample_dpm_solver_t3(model_fn, ns32, x_T, T, T_END, order=3, steps=nfe,
                                            method="singlestep")
    elif label == "rk2_t":
        rhs = lambda x, t: pf_rhs_t(eps_fn, x, t, sched=ds)
        n = max(1, nfe // 2)
        xf, real_nfe = integrate(rhs, x_T, grid_t(T, T_END, n), "midpoint")
    else:
        raise ValueError(label)
    return xf, real_nfe

SAMPLERS = ["ddim_quad", "dpm1", "dpm2", "dpm3", "dpm_fast", "rk2_t"]
GRID_NFES = [10, 12, 15, 20, 50]


## 3A. Figure 4 on our checkpoint

8 starting noises (rows within each sampler block are the same 8 seeds every
column). Columns are NFE 10, 12, 15, 20, 50, plus the 201-NFE reference. Each
tile is labelled with its own per-image error in 8-bit gray levels (an
approximate visibility scale: ~1 = invisible, 20+ = a visibly different
picture) -- `mean(|x_uint8 - ref_uint8|)` over pixels and channels.

In [ ]:
def gray_level_err(x, ref):
    a = to_uint8(x).astype(np.float64)
    b = to_uint8(ref).astype(np.float64)
    return np.abs(a - b).reshape(a.shape[0], -1).mean(axis=1)  # per-image

n_rows = len(SAMPLERS)
n_cols = len(GRID_NFES) + 1
fig, axes = plt.subplots(n_rows, n_cols, figsize=(2.0 * n_cols, 2.0 * n_rows))

for i, label in enumerate(SAMPLERS):
    for j, nfe in enumerate(GRID_NFES):
        xf, real_nfe = run_sampler(label, x_T_grid, nfe)
        gl = gray_level_err(xf, ref_grid)
        img = to_uint8(xf)[0]
        ax = axes[i, j]
        ax.imshow(img)
        ax.set_title(f"{real_nfe} NFE\ngl={gl[0]:.1f}", fontsize=8)
        ax.axis("off")
        if j == 0:
            ax.text(-0.25, 0.5, label, transform=ax.transAxes, rotation=90,
                    va="center", ha="center", fontsize=10)
    ax = axes[i, -1]
    ax.imshow(to_uint8(ref_grid)[0])
    ax.set_title("ref (201)", fontsize=8)
    ax.axis("off")

fig.suptitle(f"Tier 3, {MODEL_ID}: samples across solvers and NFE (guide's Figure-4 analogue)", y=1.0)
fig.tight_layout()
fig.savefig(FIGURES / "11_samples_grid.png", dpi=140)
plt.show()


## 3B. Per-image L2, over 64 images

M9 summed L2 over the whole batch into one number per (solver, NFE). A box
plot per sampler over `N_L2_SAMPLES` independent images shows whether that
number was one typical image or a few outliers dragging the batch mean.

In [ ]:
NFE_BUDGETS_B = [10, 12, 15, 20, 30, 50, 80, 120]
per_image_rows = []
for label in SAMPLERS:
    for nfe in NFE_BUDGETS_B:
        xf, real_nfe = run_sampler(label, x_T_l2, nfe)
        errs = np.array([l2(xf[k].cpu().numpy().ravel(), ref_l2_full[k].cpu().numpy().ravel())
                         for k in range(xf.shape[0])])
        for e in errs:
            per_image_rows.append(dict(sampler=label, nfe=real_nfe, err_l2=float(e)))

per_image = pd.DataFrame(per_image_rows)
per_image.to_csv(RESULTS / "tier3_per_image_l2.csv", index=False)

fig, ax = plt.subplots(figsize=(10, 5.5))
show_nfe = 12
sub = per_image[per_image.nfe.sub(show_nfe).abs() <= 2]
sub.boxplot(column="err_l2", by="sampler", ax=ax, rot=30)
ax.set_ylabel(f"per-image L2 (NFE ~ {show_nfe})")
ax.set_title(f"M11: per-image error spread, {N_L2_SAMPLES} images, ~{show_nfe} NFE")
plt.suptitle("")
fig.tight_layout()
fig.savefig(FIGURES / "11_per_image_l2.png", dpi=140)
plt.show()


## 3C. FID anchor

5k samples per configuration (`N_FID_SAMPLES` -- raise to 10k for a less
biased estimate, roughly doubling this section's runtime), at 10 and 20 NFE,
for every sampler above. Absolute FIDs will read higher than the paper's
(5k/10k samples here vs 50k there); compare **rankings**, not magnitudes.

Uses `clean-fid`'s precomputed CIFAR-10 train statistics
(`mode="legacy_tensorflow"`, for comparability with published numbers). If
that download fails (no internet, or the stats server is unreachable), falls
back to computing statistics from `torchvision`'s own CIFAR-10 train split.

In [ ]:
N_FID_SAMPLES = 5000   # the open decision from the M11 spec -- raise to 10000 if you have the time
FID_NFES = [10, 20]
FID_BATCH = 100

import tempfile
from PIL import Image
from cleanfid import fid as cleanfid

def save_batch_as_png(x_uint8, outdir, start_idx):
    for k in range(x_uint8.shape[0]):
        Image.fromarray(x_uint8[k]).save(outdir / f"{start_idx + k:06d}.png")

def compute_fid_for(label, nfe, n_samples):
    with tempfile.TemporaryDirectory() as tmp:
        outdir = pathlib.Path(tmp)
        done = 0
        torch.manual_seed(1000 + hash(label) % 1000)
        while done < n_samples:
            b = min(FID_BATCH, n_samples - done)
            x_T = torch.randn(b, 3, 32, 32, device=DEVICE)
            xf, real_nfe = run_sampler(label, x_T, nfe)
            save_batch_as_png(to_uint8(xf), outdir, done)
            done += b
        try:
            score = cleanfid.compute_fid(str(outdir), dataset_name="cifar10", dataset_res=32,
                                         dataset_split="train", mode="legacy_tensorflow",
                                         device=DEVICE, verbose=False)
            method = "clean-fid CIFAR-10 train stats"
        except Exception as exc:
            print(f"  !! precomputed stats failed ({exc}); falling back to torchvision CIFAR-10")
            import torchvision
            real_dir = pathlib.Path(tmp).parent / "cifar10_train_real"
            if not real_dir.exists():
                real_dir.mkdir()
                train = torchvision.datasets.CIFAR10(root=str(RESULTS / "_cifar10_cache"),
                                                      train=True, download=True)
                for k in range(min(10000, len(train))):
                    train[k][0].save(real_dir / f"{k:06d}.png")
            score = cleanfid.compute_fid(str(outdir), str(real_dir), mode="legacy_tensorflow",
                                         device=DEVICE, verbose=False)
            method = "torchvision CIFAR-10 train (fallback)"
        return score, real_nfe, method

fid_rows = []
CSV_FID = RESULTS / "tier3_fid.csv"
CSV_FID.unlink(missing_ok=True)
t0 = time.time()
for label in SAMPLERS:
    for nfe in FID_NFES:
        score, real_nfe, method = compute_fid_for(label, nfe, N_FID_SAMPLES)
        fid_rows.append(dict(config=label, nfe=real_nfe, n_samples=N_FID_SAMPLES,
                             fid=score, method=method))
        print(f"  {label:<10s} nfe={real_nfe:<4d} fid={score:7.2f}  [{time.time()-t0:6.1f}s]")

# the converged baseline
score, real_nfe, method = compute_fid_for("dpm3", 99, N_FID_SAMPLES)  # steps=99 -> nfe=99, close to 'converged'
fid_rows.append(dict(config="dpm3_converged", nfe=real_nfe, n_samples=N_FID_SAMPLES, fid=score, method=method))
print(f"  dpm3 (converged, ~100 NFE) fid={score:7.2f}")

fid_df = pd.DataFrame(fid_rows)
fid_df.to_csv(CSV_FID, index=False)
fid_df


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5.5))
piv = fid_df[fid_df.config != "dpm3_converged"].pivot(index="config", columns="nfe", values="fid")
piv.plot(kind="bar", ax=ax)
ax.set_ylabel("FID (5k samples, legacy_tensorflow)")
ax.set_title("M11: FID ranking at practitioner NFE")
ax.tick_params(axis="x", rotation=30)
fig.tight_layout()
fig.savefig(FIGURES / "11_fid_vs_l2.png", dpi=140)
plt.show()

at10 = fid_df[(fid_df.nfe.sub(10).abs() <= 1) & (fid_df.config != "dpm3_converged")].sort_values("fid")
print("FID ranking near 10 NFE (best first):")
print(at10[["config", "nfe", "fid"]].to_string(index=False))
print()
print("paper's Table 6 at 10 NFE: DPM-fast 6.42 < DPM-2 7.90 < DPM-1 16.69 ~ DDIM < DPM-3 24.37")


## 4. Decision

Fill this in from the table above.

In [ ]:
print("FID ranking at ~10 NFE (this run):", list(at10["config"]))
print()
print("If this matches DPM-fast <~ DPM-2 < DDIM ~ DPM-1 < DPM-3 (the paper's own")
print("ranking): Tier 3 is validated. The L2 ranking (from results/tier3_error.csv,")
print("M9) disagreeing with this FID ranking at the SAME NFE is the headline result --")
print("proceed to M12/M13.")
print()
print("If it does not match: stop. Debug the Tier-3 setup (discrete-schedule")
print("conversion, continuous-to-discrete time mapping) before M12/M13.")


## 5. Validation -- run the unit tests

In [ ]:
out = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_tier3.py", "-q"],
    cwd=ROOT, capture_output=True, text=True,
)
print(out.stdout); print(out.stderr)
assert out.returncode == 0, "M11 / Tier-3 tests failed"

for f in ["results/tier3_fid.csv", "results/tier3_per_image_l2.csv",
          "figures/11_samples_grid.png", "figures/11_per_image_l2.png",
          "figures/11_fid_vs_l2.png"]:
    assert (ROOT / f).exists(), f"missing deliverable: {f}"
    print(f"  ok  {f}")

print("\nM11 validation: PASS")
print("\nDownload results/tier3_fid.csv, results/tier3_per_image_l2.csv and")
print("figures/11_*.png back into the repo, then commit.")
